# Silver Layer - CRM Products

This notebook transforms `workspace.bronze.crm_prd_info_raw` into a curated Silver products table: `workspace.silver.crm_products`.

## Tools and Operations
- **PySpark DataFrame API** and **Window functions**
- Column renaming and string cleanup
- Product line normalization and key parsing
- Duplicate removal and null checks

## Output
- `workspace.silver.crm_products`


## 1) Setup


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from itertools import chain

## 2) Load and Inspect Source Data


In [0]:
df_raw = spark.table("workspace.bronze.crm_prd_info_raw")
df_raw.display()

In [0]:
print(f'Table contains {df_raw.count()} rows')
print(f'Table contains {df_raw.select("prd_id").distinct().count()} distinct product ids')
print(f'Table contains {df_raw.select("prd_key").distinct().count()} distinct product key')
print(f'Table contains {df_raw.select("prd_nm").distinct().count()} distinct product name')

In [0]:
df_raw.schema.fields

In [0]:
df_raw.select("prd_id").distinct().count()

## 3) Transformations


### Rename Columns


In [0]:
column_map= {
  'prd_id':'product_id', 
  'prd_key':'product_key', 
  'prd_nm':'product_name', 
  'prd_cost':'product_cost', 
  'prd_line':'product_line', 
  'prd_start_dt':'start_date',
  'prd_end_dt':'end_date'
  }

df = df_raw.select([F.col(c).alias(column_map.get(c, c)) for c in df_raw.columns])

df.limit(1).display()

### Trim String Columns


In [0]:
 df.schema.fields

In [0]:
df = df.select([
    F.when(F.trim(F.col(f.name)) == '', F.lit(None)).otherwise(F.trim(F.col(f.name))).alias(f.name) 
    if isinstance(f.dataType, StringType) 
    else F.col(f.name)
    for f in df.schema.fields
])

df.display()

In [0]:
print(f'Table contains {df.count()} rows')
print(f'Table contains {df.select("product_id").distinct().count()} distinct product ids')
print(f'Table contains {df.select("product_key").distinct().count()} distinct product key')
print(f'Table contains {df.select("product_name").distinct().count()} distinct product name')

### Remove Duplicate Product Key


In [0]:
w = Window.partitionBy("product_key").orderBy(F.col("start_date").desc_nulls_last())

df = (
    df
    .withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

print(f'Table contains {df.count()} rows')
print(f'Table contains {df.select("product_id").distinct().count()} distinct product ids')
print(f'Table contains {df.select("product_key").distinct().count()} distinct product key')
print(f'Table contains {df.select("product_name").distinct().count()} distinct product name')
df.display()

### Null Checks


In [0]:
df.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]).display()

### Normalize Encoded Values


In [0]:
df.select("product_line").distinct().display()


In [0]:
prod_line_map = {
    "r" : "Road" ,
    "s" : "Other Sales",
    "m" : "Mountain",
    "t" : "Touring"
}

def map_abbrev_column(df, str_col, mapping):
    mapping = {k.strip().lower() : v for k, v in mapping.items()}
    map_expr = F.create_map([F.lit(x) for x in chain(*mapping.items())])
    key = F.lower(F.trim(F.col(str_col)))
    return df.withColumn(str_col, F.coalesce(map_expr[key], key))

df = map_abbrev_column(df, "product_line", prod_line_map)

df.select("product_line").distinct().display()

### Parse Product Key


In [0]:
df = (
    df
    .withColumn("category_key", F.substring(F.col("product_key"), 1, 5))
    .withColumn("product_key", F.substring(F.col("product_key"), 7, F.length(F.col("product_key"))))
)

df.limit(5).display()

In [0]:
df.select("category_key").distinct().display()

## 4) Write Silver Table


In [0]:
%sql
DROP TABLE IF EXISTS workspace.silver.crm_products

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_products")

## 5) Validation


In [0]:
%sql
SELECT * FROM workspace.silver.crm_products